Hi! This is just a quick potential backend just to get around the machine learning issue. I'd appreciate any thoughts and if this would possibly work instead, its a very short program even with these features so we can always extend it but it could be a good demonstration for when we have to show what we have done and can be iteration 1 so that way we can show even more progress since this is fairly basic and just to experiment around with but it is fully working with the test cases I tried (pixel hearts lmao). I've connected it to desmos so any equation typed in there can be converted into a graph on our pins. The commands to use this in terminal are below the code and can take a downloaded image or an image url and by all means pull this code and experiment around / improve this. A future iteration could be to connect this to a better front end that is accessible for visually impaired students with a screenreader and high contrast as well as usuable for teachers and more collaborative and connecting this with the app.

In order to run this code you need to download Python as well as a few libraries, Claude can help you trouble shoot but as a guide these are the instructions to run on your terminal

In [ ]:
pip install playwright pillow requests
playwright install chromium

If it doesnt run the most common reason is because its in the wrong folder lmao, if you type in CD /(the download path of the file) into your terminal and then run it then it should work

In [ ]:
"""
Pin-matrix maker: 7 columns x 6 rows (42 pins)

  python3 matrix_maker.py                       live mode: type expressions in a Desmos window
  python3 matrix_maker.py --image logo.png      convert an image file
  python3 matrix_maker.py --url https://...     convert an image from the internet

Image options:
  --fill 0.4    fraction of pins that turn on (0.2 = sparse, 0.6 = heavy)
  --invert      shape is light on a dark background
  --edges       outline the shape instead of filling it

Setup:  pip install playwright pillow requests   then   playwright install chromium
Output: 0 = pin on (line / dark), 1 = pin off (background); '#' = on, '.' = off in the preview.
"""
import io
import sys
import base64
import argparse
from PIL import Image, ImageChops, ImageFilter, ImageOps

COLS, ROWS = 7, 6
IMG_W, IMG_H = COLS * 100, ROWS * 100          # 700 x 600, 100 px per cell

# Desmos window: one math unit per cell. y = 4 - x^2 fits with its vertex at top centre.
BOUNDS = (-3.5, 3.5, -1.5, 4.5)                # left, right, bottom, top
MIN_COVERAGE = 0.04                            # fraction of a cell that must be inked (Desmos mode)

SETUP_JS = """
([l, r, b, t]) => {
    Calc.setBlank();
    Calc.updateSettings({ showGrid: false, showAxes: false, squareAxes: false });
    Calc.setMathBounds({ left: l, right: r, bottom: b, top: t });
    Calc.updateSettings({ lockViewport: true });
}
"""

SHOT_JS = """
([l, r, b, t, w, h]) => new Promise(resolve => {
    Calc.asyncScreenshot(
        { width: w, height: h, targetPixelRatio: 1,
          mathBounds: { left: l, right: r, bottom: b, top: t } },
        resolve
    );
})
"""


# ---------- output ----------

def render(matrix, header="", clear=False):
    if clear:
        sys.stdout.write("\033[H\033[J")
    if header:
        print(header)
    print(f"\nGenerated {COLS}x{ROWS} Array:")
    for row in matrix:
        print(row)
    print("\nVisual preview:")
    for row in matrix:
        print(" ".join("." if v == 1 else "#" for v in row))
    sys.stdout.flush()


# ---------- Desmos screenshot -> matrix (thin coloured lines on white) ----------

def desmos_matrix(img, threshold=200, min_coverage=MIN_COVERAGE):
    rgba = img.convert("RGBA").resize((IMG_W, IMG_H))
    white = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
    white.alpha_composite(rgba)                                   # flatten transparency onto white
    r, g, b = white.convert("RGB").split()
    darkest = ImageChops.darker(ImageChops.darker(r, g), b)       # any colour of line counts as ink
    ink = darkest.point(lambda v: 255 if v < threshold else 0)
    ink = ink.filter(ImageFilter.MaxFilter(11))                   # thicken: Desmos' default line is only ~2-3 px wide

    bw, bh = IMG_W // COLS, IMG_H // ROWS
    matrix = []
    for y in range(ROWS):
        row = []
        for x in range(COLS):
            block = ink.crop((x * bw, y * bh, (x + 1) * bw, (y + 1) * bh))
            coverage = block.histogram()[255] / (bw * bh)
            row.append(0 if coverage >= min_coverage else 1)
        matrix.append(row)
    return matrix


# ---------- any photo / picture -> matrix ----------

def smart_matrix(img, fill=0.4, invert=False, edges=False):
    img = img.convert("RGB")
    img = ImageOps.fit(img, (IMG_W, IMG_H), Image.Resampling.LANCZOS)   # centre-crop to 7:6
    g = ImageOps.autocontrast(ImageOps.grayscale(img), cutoff=2)        # stretch contrast
    if edges:
        g = ImageOps.invert(g.filter(ImageFilter.FIND_EDGES))
    if invert:
        g = ImageOps.invert(g)
    small = g.resize((COLS, ROWS), Image.Resampling.BOX)                # average brightness per cell
    vals = sorted(small.tobytes())
    cut = vals[max(0, min(len(vals) - 1, int(len(vals) * fill)))]       # keep the darkest `fill` fraction
    return [[0 if small.getpixel((x, y)) < cut else 1 for x in range(COLS)] for y in range(ROWS)]


# ---------- live Desmos mode ----------

def run_live():
    from playwright.sync_api import sync_playwright, Error as PlaywrightError

    left, right, bottom, top = BOUNDS
    header = "Type an expression in the Desmos window. Close the window or press Ctrl+C to quit."

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        page = browser.new_page(viewport={"width": 1300, "height": 850})
        page.goto("https://www.desmos.com/calculator")
        page.wait_for_function("typeof Calc !== 'undefined'")
        page.evaluate(SETUP_JS, [left, right, bottom, top])

        last_state = None
        try:
            while not page.is_closed():
                state = page.evaluate("() => JSON.stringify(Calc.getState())")
                if state != last_state:
                    last_state = state
                    page.wait_for_timeout(300)                  # let Desmos finish drawing
                    data_url = page.evaluate(SHOT_JS, [left, right, bottom, top, IMG_W, IMG_H])
                    img = Image.open(io.BytesIO(base64.b64decode(data_url.split(",", 1)[1])))
                    render(desmos_matrix(img), header, clear=True)
                page.wait_for_timeout(400)
        except (PlaywrightError, KeyboardInterrupt):
            pass                                                # window closed or Ctrl+C
        finally:
            try:
                browser.close()
            except Exception:
                pass


# ---------- main ----------

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Make a 7x6 pin matrix from Desmos or any image.")
    parser.add_argument("--image", help="path to an image file")
    parser.add_argument("--url", help="URL of an image on the internet")
    parser.add_argument("--fill", type=float, default=0.4, help="fraction of pins on, 0-1 (default 0.4)")
    parser.add_argument("--invert", action="store_true", help="shape is light on a dark background")
    parser.add_argument("--edges", action="store_true", help="outline the shape instead of filling it")
    args = parser.parse_args()

    if args.url:
        import requests
        resp = requests.get(args.url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        picture = Image.open(io.BytesIO(resp.content))
        render(smart_matrix(picture, args.fill, args.invert, args.edges), f"URL: {args.url}")
    elif args.image:
        picture = Image.open(args.image)
        render(smart_matrix(picture, args.fill, args.invert, args.edges), f"Image: {args.image}")
    else:
        run_live()


If you want to experiment around with different size grids you can just change the number of cols and rows: comand F the following line 

In [ ]:
COLS, ROWS = 7, 6

You can also change the image bounds (i.e change the view window) for the desmos api using the below settings. When running it should show a debug screenshot image as one of the files so you can take a look at the screenshot of the graph that desmos took and debug / experiment that way (though i made some changes so not sure if that still works) but for view window the code to change is:

If you want to show the graph axis then simply change false to true for the following line

In [ ]:
BOUNDS = (-3.5, 3.5, -1.5, 4.5)                # left, right, bottom, top
MIN_COVERAGE = 0.04                            # fraction of a cell that must be inked (Desmos mode)

SETUP_JS = """
([l, r, b, t]) => {
    Calc.setBlank();
    Calc.updateSettings({ showGrid: false, showAxes: false, squareAxes: false });
    Calc.setMathBounds({ left: l, right: r, bottom: b, top: t });
    Calc.updateSettings({ lockViewport: true });
}
"""